### **Import libraries**

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import scanpy as sc
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

### Load Geneformer model

In [ ]:
# Load Geneformer model
model_name = "biosustain/geneformer"
model = AutoModel.from_pretrained(model_name)

### Load the preprocessed dataset

In [ ]:
# Load NSCLC gene expression dataset (GSE81089)
df = pd.read_csv("preprocessed_dataset.csv") 
labels = df["diagnosis"]  # squamous cell cancer, 2=AC unspecified, 3=Large cell/ NOS
gene_data = df.drop(columns=["sample_id","diagnosis","stage"])

### List of highly expressed genes 

In [ ]:
# For each sample, get a list of gene names (e.g., top expressed genes)
# You can choose top N genes if needed, or just take all genes
top_genes = gene_data.apply(lambda x: x.nlargest(len(x)).index.tolist(), axis=1)

# Convert gene lists into a text representation (for Geneformer)
gene_texts = [" ".join(genes) for genes in top_genes]

# Example of the first few texts
print(gene_texts[:5])

### Tokenization and Feature Extraction Using Geneformer

In [ ]:
# Load Geneformer model and tokenizer
model_name = "biosustain/geneformer"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Tokenize the gene expression data (text format)
inputs = tokenizer(gene_texts, return_tensors="pt", padding=True, truncation=True)

# Pass the data through Geneformer to get embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Use mean pooling of the last hidden state for embeddings
embeddings = outputs.last_hidden_state.mean(dim=1)

# Now, `embeddings` contains a vector representation for each sample
print(embeddings.shape)  # Check the size of the embedding vectors

### Classification

In [ ]:
# Example: Let's use `stage` as the target for classification
labels = df['stage']

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(embeddings.numpy(), labels, test_size=0.2, random_state=42)

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

### Identify Oncogenes Using Geneformer Embeddings

Step 1: Prepare Oncogene Data

ref: 
- El-Telbany A, Ma PC. Cancer genes in lung cancer: racial disparities: are there any? Genes Cancer. 2012 Jul;3(7-8):467-80. doi: 10.1177/1947601912465177. PMID: 23264847; PMCID: PMC3527990. https://pmc.ncbi.nlm.nih.gov/articles/PMC3527990/#section2-1947601912465177

In [ ]:
# List of known oncogenes
oncogenes = ["EGFR", "KRAS", "MET", "LKB1", "BRAF", "PIK3CA", "ALK", "RET", "ROS1"]

Step 2: Extract Geneformer Embeddings

Get label genes as oncogenes (1) or non-oncogenes (0)

In [ ]:
# Assuming embeddings were extracted and stored in `embeddings`
# Let's link them to the oncogene labels
labels = df.columns[df.columns.isin(oncogenes)].map(lambda gene: 1 if gene in oncogenes else 0)

# Check the shape of the embeddings and labels
print(embeddings.shape, labels.shape)

Step 3: Train a Classifier

Use the embeddings to train a classifier to predict whether a gene is an oncogene or not

In [ ]:
# Assuming embeddings and labels are aligned
X_train, X_test, y_train, y_test = train_test_split(embeddings.numpy(), labels, test_size=0.2, random_state=42)

# Train a Random Forest classifier
clf = RandomForestClassifier(n_estimators=100)
clf.fit(X_train, y_train)

# Make predictions
y_pred = clf.predict(X_test)

# Evaluate model performance
print("Accuracy:", accuracy_score(y_test, y_pred))

Step 4: Evaluate and Interpret Results

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Get probabilities for the positive class
y_prob = clf.predict_proba(X_test)[:, 1]

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right')
plt.show()


In [ ]:
import shap

# Create a SHAP explainer object for your classifier
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test)

# Plot summary plot (global feature importance)
shap.summary_plot(shap_values[1], X_test, feature_names=df.columns[1:])  # Using embeddings for interpretation